# Project 2 — Extended Solution: Modular Arithmetic Toolkit

**Python:** modern Python 3.11+ • **Dependencies:** standard library only • **Run:** Kernel → Restart & Run All

## 1. The extension's contract

**This is a standalone solution**: it does not import or depend on the first notebook. It intentionally moves beyond the original mutable class into a small, safe mathematical value-object library.

- `Mod` is a frozen, slotted dataclass. Residues are canonical, and equivalent `Mod` instances have equal hashes.
- `Mod(1, 5) != 6` **by design**. Compare integer congruence with `Mod(1, 5).congruent_to(6)` instead. This avoids the impossible hash requirement that would arise if one object compared equal to different integers.
- `<` compares canonical representatives **only for identical moduli**; no natural ordering is defined between modular rings or ordinary integers.
- `+`, `-`, `*`, division `/`, and `**` support compatible residues; arithmetic operators also accept integers, with correct reflected dispatch. With immutable objects, `+=` naturally rebinds a name rather than mutating a shared object.
- New mathematics: inverse detection, modular division, generalized Chinese Remainder Theorem (including compatible noncoprime moduli), and all solutions to linear congruences.
- `to_dict`/`from_dict` and JSON helpers provide portable serialization. `solve_linear` limits materialized output by default to avoid accidental memory exhaustion.

**Intentional difference:** `Mod` equality with an `int` in Notebook 1 is retained there for fidelity; here strict equality enables safe hashing and predictable dict/set behavior.

## 2. Complete extended implementation

The standard library supplies `dataclasses`, `math.gcd`, `operator`, and JSON. No package installation required. `NonInvertibleError` makes failed division and inverse operations easier to handle than a generic `ValueError`. Modular exponentiation uses efficient three-argument `pow` without reducing the exponent incorrectly.

In [1]:
from __future__ import annotations

import json
import math
import operator
from dataclasses import dataclass
from functools import total_ordering
from types import NotImplementedType
from typing import Callable, Mapping


class NonInvertibleError(ValueError):
    """A residue has no multiplicative inverse for the given modulus."""


@total_ordering
@dataclass(frozen=True, slots=True, eq=False)
class Mod:
    """Immutable residue with strict, hash-consistent equality.

    Only another Mod with the SAME modulus and value compares equal. Integer
    congruence is available explicitly via ``congruent_to``. Ordering compares
    canonical residues only between Mod objects with identical moduli.
    """

    value: int
    modulus: int

    def __post_init__(self) -> None:
        self._validate_integer(self.value, "value")
        self._validate_integer(self.modulus, "modulus")
        if self.modulus <= 0:
            raise ValueError("modulus must be strictly positive")
        object.__setattr__(self, "value", self.value % self.modulus)

    @staticmethod
    def _validate_integer(value: object, name: str) -> None:
        if not isinstance(value, int) or isinstance(value, bool):
            raise TypeError(f"{name} must be an integer (excluding bool)")

    def __hash__(self) -> int:
        return hash((self.value, self.modulus))

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Mod):
            return NotImplemented
        return (self.value, self.modulus) == (other.value, other.modulus)

    def __lt__(self, other: object) -> bool:
        if not isinstance(other, Mod) or self.modulus != other.modulus:
            return NotImplemented
        return self.value < other.value

    def __int__(self) -> int:
        return self.value

    def congruent_to(self, integer: int) -> bool:
        """Check congruence with an integer without conflating equality."""
        self._validate_integer(integer, "integer")
        return integer % self.modulus == self.value

    def _coerce(self, other: object) -> int | NotImplementedType:
        if isinstance(other, Mod):
            return other.value if other.modulus == self.modulus else NotImplemented
        if isinstance(other, int) and not isinstance(other, bool):
            return other % self.modulus
        return NotImplemented

    def _binary(
        self, other: object, op: Callable[[int, int], int], *, reverse: bool = False
    ) -> Mod | NotImplementedType:
        value = self._coerce(other)
        if value is NotImplemented:
            return NotImplemented
        a, b = (value, self.value) if reverse else (self.value, value)
        return Mod(op(a, b), self.modulus)

    def __neg__(self) -> Mod:
        return Mod(-self.value, self.modulus)

    def __pos__(self) -> Mod:
        return self  # immutable

    def __add__(self, other: object) -> Mod:
        return self._binary(other, operator.add)

    def __radd__(self, other: object) -> Mod:
        return self._binary(other, operator.add, reverse=True)

    def __sub__(self, other: object) -> Mod:
        return self._binary(other, operator.sub)

    def __rsub__(self, other: object) -> Mod:
        return self._binary(other, operator.sub, reverse=True)

    def __mul__(self, other: object) -> Mod:
        return self._binary(other, operator.mul)

    def __rmul__(self, other: object) -> Mod:
        return self._binary(other, operator.mul, reverse=True)

    @property
    def is_unit(self) -> bool:
        """Whether this residue has a multiplicative inverse."""
        return math.gcd(self.value, self.modulus) == 1

    def inverse(self) -> Mod:
        """Find a multiplicative inverse; raise for non-units."""
        if not self.is_unit:
            raise NonInvertibleError(
                f"{self.value} is not invertible modulo {self.modulus}; "
                f"gcd={math.gcd(self.value, self.modulus)}"
            )
        return Mod(pow(self.value, -1, self.modulus), self.modulus)

    def __truediv__(self, other: object) -> Mod:
        divisor = self._coerce(other)
        if divisor is NotImplemented:
            return NotImplemented
        return self * Mod(divisor, self.modulus).inverse()

    def __rtruediv__(self, other: object) -> Mod:
        numerator = self._coerce(other)
        if numerator is NotImplemented:
            return NotImplemented
        return Mod(numerator, self.modulus) * self.inverse()

    def __pow__(self, exponent: object, modulo: object = None) -> Mod:
        if modulo is not None:
            raise TypeError("Mod already defines the modulus; use base ** exponent")
        if isinstance(exponent, Mod):
            if exponent.modulus != self.modulus:
                return NotImplemented
            exp = exponent.value
        elif isinstance(exponent, int) and not isinstance(exponent, bool):
            exp = exponent
        else:
            return NotImplemented
        if exp < 0 and not self.is_unit:
            raise NonInvertibleError("negative powers require an invertible base")
        return Mod(pow(self.value, exp, self.modulus), self.modulus)

    def __rpow__(self, base: object) -> Mod:
        value = self._coerce(base)
        if value is NotImplemented:
            return NotImplemented
        return Mod(pow(value, self.value, self.modulus), self.modulus)

    def to_dict(self) -> dict[str, int]:
        """Return a minimal, canonical and JSON-serializable representation."""
        return {"value": self.value, "modulus": self.modulus}

    @classmethod
    def from_dict(cls, data: Mapping[str, object]) -> Mod:
        if not isinstance(data, Mapping) or set(data) != {"value", "modulus"}:
            raise ValueError("expected exactly the keys 'value' and 'modulus'")
        return cls(data["value"], data["modulus"])

    def to_json(self) -> str:
        return json.dumps(self.to_dict(), sort_keys=True)

    @classmethod
    def from_json(cls, text: str) -> Mod:
        return cls.from_dict(json.loads(text))

    @staticmethod
    def crt(*congruences: Mod) -> Mod:
        """Generalized Chinese Remainder Theorem (noncoprime moduli allowed).

        Raise ValueError when congruences are mutually inconsistent. The result
        is the unique solution modulo the least common multiple of all moduli.
        """
        if not congruences:
            raise ValueError("supply at least one congruence")
        if any(not isinstance(item, Mod) for item in congruences):
            raise TypeError("all congruences must be Mod instances")
        residue, modulus = congruences[0].value, congruences[0].modulus
        for equation in congruences[1:]:
            b, n = equation.value, equation.modulus
            g = math.gcd(modulus, n)
            difference = b - residue
            if difference % g:
                raise ValueError("inconsistent congruences: no common solution")
            n_reduced = n // g
            factor = (
                ((difference // g) * pow(modulus // g, -1, n_reduced)) % n_reduced
                if n_reduced > 1 else 0
            )
            residue = (residue + modulus * factor) % (modulus * n_reduced)
            modulus *= n_reduced
        return Mod(residue, modulus)

    @staticmethod
    def solve_linear(
        a: int, b: int, modulus: int, *, max_solutions: int = 10_000
    ) -> tuple[Mod, ...]:
        """Find all x in [0, modulus) satisfying a*x ≡ b (mod modulus).

        A solution exists iff gcd(a, modulus) divides b. The explicit result
        bound avoids unexpectedly materializing millions of solutions.
        """
        for name, value in (("a", a), ("b", b), ("modulus", modulus),
                            ("max_solutions", max_solutions)):
            Mod._validate_integer(value, name)
        if modulus <= 0 or max_solutions <= 0:
            raise ValueError("modulus and max_solutions must be positive")
        g = math.gcd(a, modulus)
        if b % g:
            return ()
        if g > max_solutions:
            raise ValueError("too many solutions to materialize; increase max_solutions")
        reduced_modulus = modulus // g
        start = (
            ((b // g) * pow(a // g, -1, reduced_modulus)) % reduced_modulus
            if reduced_modulus > 1 else 0
        )
        return tuple(Mod(start + k * reduced_modulus, modulus) for k in range(g))


## 3. Practical examples

Inverse/division work only for *units*, the residues coprime to their modulus. Generalized CRT returns one solution modulo the least common multiple, or raises for inconsistent constraints. Solving a linear congruence can produce zero, one, or multiple results.

In [2]:
a = Mod(3, 11)
print("integer congruence:", a.congruent_to(14))
print("inverse:", a.inverse(), "division:", Mod(6, 11) / a)
print("immutable and hashable:", {Mod(6, 5): "stored"}[Mod(1, 5)])
print("CRT, noncoprime:", Mod.crt(Mod(2, 6), Mod(8, 9)))
print("linear solutions to 4*x = 2 (mod 6):", Mod.solve_linear(4, 2, 6))
print("JSON round trip:", Mod.from_json(a.to_json()))


integer congruence: True
inverse: Mod(value=4, modulus=11) division: Mod(value=2, modulus=11)
immutable and hashable: stored
CRT, noncoprime: Mod(value=8, modulus=18)
linear solutions to 4*x = 2 (mod 6): (Mod(value=2, modulus=6), Mod(value=5, modulus=6))
JSON round trip: Mod(value=3, modulus=11)


## 4. Proof sketches and complexity

**Inverse:** `a` has an inverse modulo `n` iff `gcd(a, n) == 1`. Python's `pow(a, -1, n)` computes one efficiently. **Division:** `a / b` means `a * inverse(b)` and is intentionally unavailable when `b` is not invertible. **CRT merge:** for `x ≡ a (mod m)` and `x ≡ b (mod n)`, let `g=gcd(m,n)`. Existence requires `(b-a) % g == 0`; solve for `k` in `x=a+m*k` modulo `n/g`, then normalize modulo `lcm(m,n)`. **Linear congruence:** `a*x ≡ b (mod n)` has solutions iff `gcd(a,n)` divides `b`; in that case there are precisely `gcd(a,n)` distinct solutions modulo `n`.

Modular exponentiation runs in logarithmically many multiplication steps relative to the exponent, whereas listing *all* solutions necessarily takes time and memory proportional to the number of results (hence the explicit bound). This is a learning library, not a constant-time cryptographic implementation: do not use it to process private cryptographic keys.

## 5. Comprehensive `unittest` suite

Beyond example-based assertions, these tests perform exhaustive small-domain checks against Python integer arithmetic, brute-force CRT, brute-force linear equations, and inverse properties. Any test failure raises at the end of this cell, making the notebook suitable for automated execution.

In [3]:
import dataclasses
import unittest


class TestModExtended(unittest.TestCase):
    def test_immutable(self):
        item = Mod(1, 7)
        with self.assertRaises((dataclasses.FrozenInstanceError, AttributeError)):
            item.value = 3
        with self.assertRaises((dataclasses.FrozenInstanceError, AttributeError)):
            item.modulus = 5

    def test_canonical_value(self):
        self.assertEqual(Mod(-17, 5), Mod(3, 5))

    def test_reject_bool_and_bad_modulus(self):
        with self.assertRaises(TypeError):
            Mod(True, 5)
        with self.assertRaises(ValueError):
            Mod(1, 0)

    def test_strict_equality(self):
        self.assertNotEqual(Mod(1, 5), 6)
        self.assertNotEqual(Mod(1, 5), Mod(1, 7))
        self.assertEqual(Mod(1, 5), Mod(6, 5))

    def test_explicit_integer_congruence(self):
        self.assertTrue(Mod(1, 5).congruent_to(11))
        self.assertFalse(Mod(1, 5).congruent_to(12))

    def test_hash_and_dict_keys(self):
        data = {Mod(6, 5): "yes"}
        self.assertEqual(data[Mod(1, 5)], "yes")
        self.assertEqual(len({Mod(1, 5), Mod(6, 5), Mod(1, 7)}), 2)

    def test_hash_contract_with_ints(self):
        self.assertNotEqual(Mod(1, 5), 1)
        self.assertNotEqual(1, Mod(1, 5))

    def test_ordering_restricted_to_same_modulus(self):
        self.assertTrue(Mod(1, 5) < Mod(3, 5))
        with self.assertRaises(TypeError):
            Mod(1, 5) < 3

    def test_inplace_rebind_is_immutable(self):
        original = Mod(1, 5)
        alias = original
        original += 2
        self.assertEqual(alias.value, 1)
        self.assertEqual(original.value, 3)
        self.assertIsNot(original, alias)

    def test_arithmetic_and_reflection(self):
        a = Mod(3, 7)
        self.assertEqual(a + 6, Mod(2, 7))
        self.assertEqual(6 + a, Mod(2, 7))
        self.assertEqual(6 - a, Mod(3, 7))
        self.assertEqual(a - 6, Mod(4, 7))
        self.assertEqual(5 * a, Mod(1, 7))
        self.assertEqual(-a, Mod(4, 7))

    def test_nonmatching_modulus(self):
        with self.assertRaises(TypeError):
            Mod(1, 5) + Mod(1, 7)

    def test_nonunit_inverse(self):
        self.assertFalse(Mod(2, 6).is_unit)
        with self.assertRaises(NonInvertibleError):
            Mod(2, 6).inverse()

    def test_inverse(self):
        self.assertEqual(Mod(3, 11).inverse(), Mod(4, 11))
        self.assertEqual(Mod(3, 11) * Mod(3, 11).inverse(), Mod(1, 11))

    def test_division(self):
        self.assertEqual(Mod(6, 11) / Mod(3, 11), Mod(2, 11))
        self.assertEqual(Mod(6, 11) / 3, Mod(2, 11))
        self.assertEqual(6 / Mod(3, 11), Mod(2, 11))

    def test_division_nonunit(self):
        with self.assertRaises(NonInvertibleError):
            Mod(1, 8) / 2

    def test_power_correctness(self):
        self.assertEqual(Mod(2, 5) ** 5, Mod(2, 5))
        self.assertEqual(Mod(2, 101) ** 10**6, Mod(pow(2, 10**6, 101), 101))

    def test_negative_powers(self):
        self.assertEqual(Mod(3, 11) ** -1, Mod(4, 11))
        with self.assertRaises(NonInvertibleError):
            Mod(2, 6) ** -1

    def test_reflected_power(self):
        self.assertEqual(2 ** Mod(3, 5), Mod(3, 5))

    def test_power_third_arg_rejected(self):
        with self.assertRaises(TypeError):
            pow(Mod(2, 5), 3, 5)

    def test_json_roundtrip(self):
        item = Mod(-4, 9)
        self.assertEqual(Mod.from_json(item.to_json()), item)
        self.assertEqual(Mod.from_dict(item.to_dict()), item)

    def test_json_invalid_data(self):
        for data in ({"value": 1}, {"value": True, "modulus": 5},
                     {"value": 1, "modulus": 5, "extra": 0}):
            with self.subTest(data=data), self.assertRaises((TypeError, ValueError)):
                Mod.from_dict(data)

    def test_crt_coprime(self):
        self.assertEqual(Mod.crt(Mod(2, 3), Mod(3, 5)), Mod(8, 15))

    def test_crt_three_equations(self):
        self.assertEqual(Mod.crt(Mod(2, 3), Mod(3, 5), Mod(2, 7)), Mod(23, 105))

    def test_crt_noncoprime_consistent(self):
        self.assertEqual(Mod.crt(Mod(2, 6), Mod(8, 9)), Mod(8, 18))

    def test_crt_noncoprime_inconsistent(self):
        with self.assertRaises(ValueError):
            Mod.crt(Mod(1, 6), Mod(2, 9))

    def test_crt_validation(self):
        with self.assertRaises(ValueError):
            Mod.crt()
        with self.assertRaises(TypeError):
            Mod.crt(Mod(1, 5), 2)

    def test_crt_modulus_one(self):
        self.assertEqual(Mod.crt(Mod(0, 1), Mod(2, 5)), Mod(2, 5))

    def test_linear_unique_solution(self):
        self.assertEqual(Mod.solve_linear(3, 4, 7), (Mod(6, 7),))

    def test_linear_multiple_solutions(self):
        self.assertEqual(Mod.solve_linear(4, 2, 6), (Mod(2, 6), Mod(5, 6)))

    def test_linear_no_solution(self):
        self.assertEqual(Mod.solve_linear(4, 3, 6), ())

    def test_linear_zero_coefficient(self):
        self.assertEqual(Mod.solve_linear(0, 0, 3),
                         (Mod(0, 3), Mod(1, 3), Mod(2, 3)))
        self.assertEqual(Mod.solve_linear(0, 1, 3), ())

    def test_linear_modulus_one(self):
        self.assertEqual(Mod.solve_linear(0, 0, 1), (Mod(0, 1),))

    def test_linear_guard(self):
        with self.assertRaises(ValueError):
            Mod.solve_linear(0, 0, 101, max_solutions=100)

    def test_exhaustive_ring_operations(self):
        for modulus in range(1, 11):
            for a in range(-5, 6):
                for b in range(-5, 6):
                    x, y = Mod(a, modulus), Mod(b, modulus)
                    self.assertEqual(x + y, Mod(a + b, modulus))
                    self.assertEqual(x - y, Mod(a - b, modulus))
                    self.assertEqual(x * y, Mod(a * b, modulus))

    def test_exhaustive_inverse_and_division(self):
        for modulus in range(2, 18):
            for a in range(modulus):
                x = Mod(a, modulus)
                if math.gcd(a, modulus) == 1:
                    self.assertEqual(x * x.inverse(), Mod(1, modulus))
                    self.assertEqual(x / x, Mod(1, modulus))
                else:
                    with self.assertRaises(NonInvertibleError):
                        x.inverse()

    def test_exhaustive_linear_solver(self):
        for modulus in range(1, 14):
            for a in range(-5, 6):
                for b in range(-5, 6):
                    expected = tuple(Mod(x, modulus) for x in range(modulus)
                                     if (a*x-b) % modulus == 0)
                    self.assertEqual(Mod.solve_linear(a, b, modulus), expected)

    def test_crt_against_brute_force(self):
        for m in range(1, 9):
            for n in range(1, 9):
                for a in range(m):
                    for b in range(n):
                        matches = [x for x in range(math.lcm(m, n))
                                   if x % m == a and x % n == b]
                        if matches:
                            self.assertEqual(Mod.crt(Mod(a, m), Mod(b, n)),
                                             Mod(matches[0], math.lcm(m, n)))
                        else:
                            with self.assertRaises(ValueError):
                                Mod.crt(Mod(a, m), Mod(b, n))


suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestModExtended)
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful(), "Extended solution tests failed"


test_arithmetic_and_reflection (__main__.TestModExtended.test_arithmetic_and_reflection) ... ok
test_canonical_value (__main__.TestModExtended.test_canonical_value) ... ok
test_crt_against_brute_force (__main__.TestModExtended.test_crt_against_brute_force) ... ok
test_crt_coprime (__main__.TestModExtended.test_crt_coprime) ... ok
test_crt_modulus_one (__main__.TestModExtended.test_crt_modulus_one) ... ok
test_crt_noncoprime_consistent (__main__.TestModExtended.test_crt_noncoprime_consistent) ... ok
test_crt_noncoprime_inconsistent (__main__.TestModExtended.test_crt_noncoprime_inconsistent) ... ok
test_crt_three_equations (__main__.TestModExtended.test_crt_three_equations) ... ok
test_crt_validation (__main__.TestModExtended.test_crt_validation) ... ok
test_division (__main__.TestModExtended.test_division) ... ok
test_division_nonunit (__main__.TestModExtended.test_division_nonunit) ... ok
test_exhaustive_inverse_and_division (__main__.TestModExtended.test_exhaustive_inverse_and_divisio

## 6. Production considerations

- **Explicit semantics:** pick either the mutable, integer-congruent class (Notebook 1) or this immutable, strict-equality class. They are alternative designs, not interchangeable drop-in types for hashed collections.
- **Modulus 1:** accepted for compatibility with the source's positive-modulus requirement. It is the one-element (zero) ring; its inverse behavior follows the underlying ring convention.
- **Resource limits:** untrusted inputs may contain enormous integers. Enforce input-size/time limits at your application's boundary. The default linear-solution limit protects only one obvious materialization path.
- **Serialization:** JSON is a data interchange format, not a proof of authenticity. Validate signatures separately if the serialized data is security-sensitive.
- **Testing:** the small-domain exhaustive checks provide strong regression coverage but do not replace large-input fuzz/property testing or formal proofs.